# Phase 4 · Попередній експеримент: звичайне донавчання DistilGPT-2

**Роль у LiteHorizon:** окремий методологічний орієнтир для **Phase 4 (адаптація моделей)**. Тут перевіряється повнопараметричне донавчання авторегресійної мовної моделі на WikiText-2. Це **не** запуск `horizon`, не LoRA і не експеримент E4, описаний у статті IT&I 2026.

**Протокол:** `distilgpt2` · WikiText-2 (`Salesforce/wikitext`, `wikitext-2-raw-v1`) · фільтр довжини тексту понад 50 символів · 2000 навчальних / 500 валідаційних записів · 2 епохи · batch size 8 · AdamW з `lr=5e-5` · обрізання/паддинг до 128 токенів · CE із маскуванням паддингу.

Виконайте комірки згори донизу. Інструкція зі встановлення, застереження та зв'язок із основним проєктом наведені в сусідньому **`README.md`**. У кінці є **архівні результати**, перенесені з вихідного ноутбука; вони не є результатами поточного запуску.

## 1. Імпорт залежностей

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
import math
from tqdm import tqdm

## 2. Ініціалізація моделей і токенізатора

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
student = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
s_params = sum(p.numel() for p in student.parameters())
print(f"Student: {s_params:,} params")

## 3. Функція втрат

In [ ]:
def fine_tuning_loss(logits, input_ids, attention_mask):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()

    shift_mask = attention_mask[:, 1:].contiguous()

    loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction="none"
    )

    loss = loss.view(shift_labels.shape)
    loss = (loss * shift_mask).sum() / shift_mask.sum()

    return loss

## 4. Навчання на WikiText-2

In [ ]:
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="train"
)
dataset = dataset.filter(lambda x: len(x['text'].strip()) > 50)

def tokenize(examples):
    return tokenizer(examples['text'], truncation=True,
        max_length=128, padding='max_length')

tokenized = dataset.select(range(2000)).map(
    tokenize, batched=True, remove_columns=['text'])
tokenized.set_format('torch')

loader = DataLoader(tokenized, batch_size=8, shuffle=True)
optimizer = torch.optim.AdamW(student.parameters(), lr=5e-5)
student.train()

for epoch in range(2):
    total_loss = 0.0

    for batch in tqdm(loader, desc=f"Epoch {epoch+1}"):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)

        optimizer.zero_grad()
        student_out = student(
            input_ids,
            attention_mask=attn
        )
        loss = fine_tuning_loss(
            student_out.logits,
            input_ids,
            attn
        )
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} avg loss: {total_loss / len(loader):.4f}")

## 5. Валідаційна perplexity

In [ ]:
def compute_perplexity(model, eval_loader):
    model.eval()
    total_nll = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in eval_loader:
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)

            labels = ids.clone()

            # Ignore padding tokens in cross-entropy
            labels[attn == 0] = -100

            out = model(
                input_ids=ids,
                attention_mask=attn,
                labels=labels
            )

            num_tokens = (labels[:, 1:] != -100).sum().item()

            total_nll += out.loss.item() * num_tokens
            total_tokens += num_tokens

    return math.exp(total_nll / total_tokens)
            
eval_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
eval_data = eval_data.filter(lambda x: len(x['text'].strip()) > 50)
eval_tok = eval_data.select(range(500)).map(
    tokenize, batched=True, remove_columns=['text'])
eval_tok.set_format('torch')
eval_loader = DataLoader(eval_tok, batch_size=8)

print(f"Fine-tuned model perplexity: {compute_perplexity(student, eval_loader):.2f}")

## Архівні результати оригінального запуску

| Показник | Значення |
|---|---:|
| Параметрів студента | 81 912 576 |
| Пристрій | CPU |
| Середня навчальна втрата, епоха 1 | 4,5960 |
| Середня навчальна втрата, епоха 2 | 3,6105 |
| Тривалість епохи 1 у збереженому журналі | 12 хв 43 с |
| Тривалість епохи 2 у збереженому журналі | 12 хв 36 с |
| **Perplexity донавченого студента** | **43,32** |

*Ці числа витягнуто зі збережених виводів старого ноутбука. Навчання не повторювалося під час очищення. В оригіналі не зафіксовано `random seed`, тому самі результати не можна вважати строго відтворюваними або статистично порівнюваними з другим ноутбуком. Значення perplexity не вимірює здатність будувати `Analysis` у Horizon.*